# Week 6 – PCA: my extra work
COS30018 Lab 6. After reading the PCA PDF and running `PCA.ipynb`, I wanted to check the ideas myself:

* **A.** a 2-feature toy example following the PDF's steps: centre the data, then find PC1 by literally trying lines through the origin and keeping the one with the **largest sum of squared distances** (SS) of the projected points – then compare with what SVD gives
* **B.** PCA "by hand" with numpy's SVD on the same student data as `PCA.ipynb` → same answer as sklearn?
* **C.** PCA on a real dataset (iris, 4 features → 2D) + loading scores
* **D.** PCA on handwritten digits (64 features → 2D) and how many PCs are needed for 95% of the variance

In [ ]:
import numpy as np
import pandas as pd
import random as rd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn import preprocessing
import os

os.makedirs("figures", exist_ok=True)
plt.rcParams["figure.dpi"] = 110

## A. Toy example – finding PC1 the way the PDF explains it
My own 6 samples with 2 features (like Gene 1 / Gene 2 in the PDF).

In [ ]:
gene1 = np.array([10, 11, 8, 3, 2, 1], dtype=float)
gene2 = np.array([6, 4, 5, 3, 2.8, 1], dtype=float)
data2 = np.column_stack([gene1, gene2])

centre = data2.mean(axis=0)               # the "blue X" in the PDF
Xc = data2 - centre                       # shift so the centre is at the origin
print("centre of the data:", centre)

# try lines through the origin at every angle, project the points onto each line
angles = np.radians(np.arange(0, 180, 0.5))
ss = []
for a in angles:
    u = np.array([np.cos(a), np.sin(a)])  # unit vector along the line
    d = Xc @ u                            # distance of each projected point from the origin
    ss.append(np.sum(d ** 2))             # SS(distances)
ss = np.array(ss)
best = angles[np.argmax(ss)]
u_best = np.array([np.cos(best), np.sin(best)])
print(f"best line: angle {np.degrees(best):.1f} deg, slope {np.tan(best):.3f}, "
      f"unit vector (eigenvector) = [{u_best[0]:.3f}, {u_best[1]:.3f}], SS = {ss.max():.2f}")

# the proper way: SVD
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
pc1 = Vt[0] * np.sign(Vt[0][0])
print(f"SVD PC1  : [{pc1[0]:.3f}, {pc1[1]:.3f}], eigenvalue (SS) = S1^2 = {S[0]**2:.2f}")
var = S ** 2 / (len(Xc) - 1)              # variation = SS / (n - 1)
print(f"variation: PC1 = {var[0]:.2f}, PC2 = {var[1]:.2f}  ->  PC1 explains {var[0] / var.sum():.1%}, PC2 {var[1] / var.sum():.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
ax = axes[0]
ax.scatter(Xc[:, 0], Xc[:, 1], color="tab:red", zorder=3, label="centred samples")
t = np.linspace(-7, 7, 2)
ax.plot(t * pc1[0], t * pc1[1], label="PC1", color="tab:blue")
pc2 = np.array([-pc1[1], pc1[0]])
ax.plot(t * pc2[0] * 0.5, t * pc2[1] * 0.5, label="PC2 (perpendicular)", color="tab:green")
proj = np.outer(Xc @ pc1, pc1)
for p, q in zip(Xc, proj):
    ax.plot([p[0], q[0]], [p[1], q[1]], "k:", lw=1)
ax.axhline(0, color="grey", lw=0.5); ax.axvline(0, color="grey", lw=0.5)
ax.set_aspect("equal"); ax.set_xlabel("Gene 1 (centred)"); ax.set_ylabel("Gene 2 (centred)")
ax.set_title("PC1 = line with the biggest SS(distances)"); ax.legend(fontsize=7)

ax = axes[1]
ax.plot(np.degrees(angles), ss)
ax.axvline(np.degrees(best), color="tab:red", ls="--", label=f"max at {np.degrees(best):.1f}°")
ax.set_xlabel("angle of the line (degrees)"); ax.set_ylabel("SS(distances)")
ax.set_title("Trying every line through the origin"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("figures/6_5_pca_toy_line_search.png", dpi=150); plt.show()

The brute-force search lands on the same direction as SVD (up to 0.5° steps) – so "PCA = the line that maximises the SS of the projected distances" is literally what it computes. Here PC1 holds ~96% of the variation, so 1 dimension would almost be enough for this data.

## B. PCA by hand (SVD) vs sklearn on the `PCA.ipynb` student data
Same code + same seed (42) as my run of `PCA.ipynb`, so it's the same data.

In [ ]:
rd.seed(42)
np.random.seed(42)
students = ['student' + str(i) for i in range(1, 101)]
maths = ['maths_' + str(i) for i in range(1, 6)]
physics = ['physics_' + str(i) for i in range(1, 6)]
data = pd.DataFrame(columns=[*maths, *physics], index=students)
for student in data.index:
    data.loc[student, 'maths_1':'maths_5'] = np.random.poisson(lam=rd.randrange(0, 100), size=5)
    data.loc[student, 'physics_1':'physics_5'] = np.random.poisson(lam=rd.randrange(0, 100), size=5)
scaled = preprocessing.scale(data.T.astype(float))      # rows = the 10 scores, columns = 100 students

sk = PCA().fit(scaled)

Xc = scaled - scaled.mean(axis=0)                       # PCA always centres first
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
my_ratio = S ** 2 / np.sum(S ** 2)

print("explained variance % (sklearn):", np.round(sk.explained_variance_ratio_[:4] * 100, 1))
print("explained variance % (my SVD) :", np.round(my_ratio[:4] * 100, 1))
print("same PC1 direction (up to sign)?", np.allclose(np.abs(sk.components_[0]), np.abs(Vt[0])))

Same numbers – sklearn's PCA is just centring + SVD. Note the sign of a PC can flip between implementations/runs (it's only a direction), which is also why my PCA graph has physics on the **right** while the notebook's text says left.

## C. PCA on iris (4 features → 2D)

In [ ]:
from sklearn.datasets import load_iris
iris = load_iris()
Xi = preprocessing.StandardScaler().fit_transform(iris.data)   # scale so cm vs cm doesn't matter
pca_i = PCA().fit(Xi)
Zi = pca_i.transform(Xi)
ratio = pca_i.explained_variance_ratio_
print("explained variance %:", np.round(ratio * 100, 1), "| PC1+PC2 =", round(ratio[:2].sum() * 100, 1), "%")

loadings = pd.DataFrame(pca_i.components_[:2].T, index=iris.feature_names, columns=["PC1", "PC2"]).round(3)
loadings

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8), gridspec_kw={"width_ratios": [1, 1.35]})
ax = axes[0]
ax.bar(range(1, 5), ratio * 100, tick_label=[f"PC{i}" for i in range(1, 5)])
ax.plot(range(1, 5), np.cumsum(ratio) * 100, "ro-", label="cumulative")
ax.set_ylabel("% of explained variance"); ax.set_title("Iris scree plot"); ax.legend(fontsize=8)
ax = axes[1]
for k, name in enumerate(iris.target_names):
    m = iris.target == k
    ax.scatter(Zi[m, 0], Zi[m, 1], s=18, alpha=0.8, label=name)
ax.set_xlabel(f"PC1 - {ratio[0]*100:.1f}%"); ax.set_ylabel(f"PC2 - {ratio[1]*100:.1f}%")
ax.set_title("Iris flowers in PC1/PC2 space"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("figures/6_6_iris_pca.png", dpi=150); plt.show()

PC1 alone separates setosa completely and PC1+PC2 keep ~96% of the information, so the 2D plot is a really good summary of the 4D data. The loading scores show PC1 is mostly petal length/width (+ sepal length) – those are the features that matter for telling the species apart.

## D. Handwritten digits (64 features → 2D)

In [ ]:
from sklearn.datasets import load_digits
digits = load_digits()
pca_d = PCA().fit(digits.data)
cum = np.cumsum(pca_d.explained_variance_ratio_)
n95 = int(np.argmax(cum >= 0.95)) + 1
print(f"{digits.data.shape[1]} features -> {n95} PCs keep 95% of the variance; 2 PCs keep {cum[1]:.1%}")

Zd = pca_d.transform(digits.data)[:, :2]
plt.figure(figsize=(5.5, 4.5))
sc = plt.scatter(Zd[:, 0], Zd[:, 1], c=digits.target, cmap="tab10", s=8, alpha=0.7)
plt.colorbar(sc, ticks=range(10), label="digit")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("8x8 digits squashed to 2D with PCA")
plt.tight_layout(); plt.savefig("figures/6_7_digits_pca.png", dpi=150); plt.show()

Only ~29% of the variance survives in 2D here, so the digits overlap a lot (0s, 4s and 6s still form their own blobs). Compare with iris where 2 PCs kept 96% – that's exactly the PDF's last point: if the scree plot is "flat", a 2D PCA plot won't represent the data well.

## Takeaways
* PCA = centre (+ usually scale) the data, find the directions of maximum spread (SVD), project onto the first few.
* Scree plot / explained variance tells you how many PCs you can get away with.
* Loading scores tell you which original features build each PC.
* PCs are only defined up to a sign, so plots can come out mirrored between runs.